# 301 · Trust boundaries experiment

This notebook goes with the article
[Trust boundaries](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/301/trust-boundaries/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/301/trust_boundaries.ipynb)

**Central question:** may these bytes use a **language-native** format (for example Python pickle),
or must they use a **portable** format that other languages and teams can safely consume?

The answer follows the **data path**—who can write the bytes, who can read them, and how long they live—
not how fast a native format looked in a microbenchmark.

> **A note on numbers:** sizes and timings in these notebooks are only illustrations. For measured library comparisons on this project’s harness, use the suite [Results](https://leo-gan.github.io/GLD.SerializerBenchmark/) pages.


## Policy over hops

Each scenario is a short path made of hops (API edge, cache, object store, queue, and so on).
For every hop, mark whether an untrusted party or another language might produce or consume the bytes,
whether the system is multi-tenant, and whether the data is long-lived.

The helper should require a **portable** format on public, multi-tenant, and cross-language paths.
The private cache used by a single service binary is the only case where a native format might still be acceptable—
and only with an explicit threat model.


In [ ]:
from dataclasses import dataclass
from typing import List


@dataclass
class Hop:
    name: str
    untrusted_or_other_lang: bool
    multi_tenant: bool
    long_lived: bool
    notes: str = ""


def policy_for_path(hops: List[Hop]) -> str:
    for h in hops:
        if h.untrusted_or_other_lang or h.multi_tenant:
            return "PORTABLE required (native disqualified on this path)"
        if h.long_lived:
            return "PORTABLE strongly preferred (native lock-in / skew risk)"
    return "NATIVE allowed only if same-runtime, trusted peers, documented threat model"


paths = {
    "public HTTP body": [Hop("edge API", True, True, False)],
    "Redis cache same binary only": [Hop("redis", False, False, False, "private net")],
    "S3 model blob multi-team": [Hop("object store", False, True, True)],
    "queue Python → Go consumer": [Hop("kafka", True, False, True)],
}
for name, hops in paths.items():
    print(f"{name:32} → {policy_for_path(hops)}")



## Size is not a trust argument

This optional cell compares JSON and pickle sizes for one object.
Pickle may be smaller. That fact still does not justify pickle on a path that failed the trust test.


In [ ]:
import json, pickle

obj = {"user_id": 1, "roles": ["admin", "ops"], "prefs": {f"k{i}": i for i in range(20)}}
j = json.dumps(obj).encode()
p = pickle.dumps(obj)
print(f"JSON {len(j)} bytes; pickle {len(p)} bytes")
print("Faster/smaller native still loses if any hop fails the trust test.")



## What to optimize for (from the article)

| Signal | Role in the decision |
|--------|----------------------|
| Does the path cross a trust domain? | **Primary** gate |
| How many language runtimes consume the bytes? | More than one family forces portable formats |
| Suite speed or size | Secondary—and only among formats the policy already allows |

A good written decision sounds like: “This queue is multi-tenant, so portable only; native pickle is rejected even if it is smaller.”
